In [57]:
import os
import openai

from langsmith import Client
from markdown_it.rules_block import reference
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path.cwd().parents[1] / ".env")

True

In [ ]:

api_key = os.getenv("LANGSMITH_API_KEY", "")
client = Client(
    api_key=api_key,
)


In [58]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [60]:
dataset


Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('9780f815-a23b-44b3-9c6c-4cbcd3ceadea'), created_at=datetime.datetime(2026, 4, 9, 21, 3, 10, 626305, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 4, 9, 21, 3, 10, 626305, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.7.26', 'runtime_version': '3.12.5', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [61]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[8].outputs

{'ground_truth': "The Tenda A33 extender (B0BZ5R7CVP) includes a Gigabit Ethernet port for demanding tasks like gaming. If you need a high‑quality cable to connect a laptop or console to the extender, the UseBean USB‑C cable (B0BP9Z159S) can provide 4K video and high‑power charging but for Ethernet you'd use a standard Cat5e/Cat6 cable (not listed). The extender's Gigabit port itself is what enables wired gaming performance.",
 'reference_context_ids': ['B0BZ5R7CVP', 'B0BP9Z159S'],
 'reference_description': ['Tenda A33 AX3000 WiFi 6 Extender, WiFi Booster WiFi Range Extender, 2.4/5GHz Dual Band WiFi Extender with Ethernet Port, AP Mode, WPS Easy Setup, WiFi Extenders Signal Booster for Home Improved WiFi Coverage - With 2 * 5dbi dual-band antennas, the Tenda A33 wifi extender can effectively boost your wifi signal for up to 2100 sq. ft coverage , providing you a stable wifi connection at your home with no dead zone. Fast Speed - Tenda A33 WiFi 6 Range Extender provides maximum speeds o

In [44]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[9].inputs

{'question': 'Combined question: Which cables and adapters in the catalog would let me connect a USB-C laptop to a 4K monitor and also provide fast charging?'}

In [62]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[9].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[9].outputs

In [29]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )


    return response.data[0].embedding



def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-item-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }



def process_context(context):
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"],
                                 context["retrieved_context_ratings"]):
        formatted_context += f"-ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context



def build_prompt(preprocessed_context, question):
    prompt = f"""
        You are a shopping assistant that can answer question about products in stock
        You will be given a question and a list of context.

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
        """

    return prompt



def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )


    return response.choices[0].message.content

def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieve_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieve_context["retrieved_context_ids"],
        "retrieved_context": retrieve_context["retrieved_context"],
        "similarity_scores": retrieve_context["similarity_scores"],
    }

    return final_result


In [63]:
rag_pipeline("can i gat some charger?", top_k=5)

{'answer': 'Yes. Here are charger cables currently available:\n\n- B0BYYLJRHT: 3-pack, 3ft iPhone/Apple MFi certified Lightning cables (Black). Includes 3 cables; compatible with most iPhone models and iPad/iPod. Durability-focused design.\n- B0BV6PWVCG: GREPHONE 2-pack, 6 ft USB-C to Lightning cables. MFi certified; supports fast charging. Includes 2 cables.\n- B0BFPZGYLD: 5 in 1/10 ft multi-charging cable set (not just for iPad). 2-pack, 6 cables in total in one 10 ft reel; supports multiple connectors (Lightning, USB-C, Micro USB). Note: designed mainly for charging; data transfer may be limited.\n- B0BGDQLZD2: Mixblu Charger Cable Replacement for Fitbit Inspire 3 (2-pack, 3.3 ft) — specific to Inspire 3.\n- B09TNXY54Y: MUXA 6-pack, various lengths (3/3/6/6/10/10 ft) USB Lightning cables with Apple MFi certification; multiple color options.\n\nIf you tell me which length and connector you need (Lightning for iPhone, USB-C, or multi-port), I can pick the best option for you.',
 'ques

### RAG Metrics

In [33]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import _IDBasedContextPrecision, _IDBasedContextRecall, _Faithfulness, _ResponseRelevancy


In [34]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

ragas_metrics = [
    _IDBasedContextPrecision(ragas_llm,ragas_embeddings),
    _IDBasedContextRecall(ragas_llm,ragas_embeddings),
    _Faithfulness(ragas_llm),
]

C:\Users\Denys\AppData\Local\Temp\ipykernel_8540\2882929943.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
C:\Users\Denys\AppData\Local\Temp\ipykernel_8540\2882929943.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [47]:
result = rag_pipeline(reference_input["question"])

In [48]:
result

{'answer': 'Based on the available products, the best match for connecting a USB-C laptop to a 4K monitor with fast charging is:\n\n- UseBean 240W USB C to USB C Cable 6.6ft (2 Pack) with 4K@60Hz video output and PD fast charging up to 240W when paired with a compatible charger. It supports 4K video output (3840x2160) and is compatible with Thunderbolt 3/4 devices and USB-C devices, and it handles fast charging (PD) when the host charger supports it.\n\nNotes:\n- This cable supports 4K video output and data transfer, and it is designed for high-wattage PD charging when used with a suitable charger. \n- You will need a USB-C charger that supports PD and matches the 240W capability (or at least a charger capable of PD fast charging for your device) to achieve the fast charging speed.\n\nNo other listed cables/adapters in the catalog explicitly combine 4K video output with PD fast charging for USB-C laptops. If you need a separate adapter to connect to the monitor (e.g., if your monitor r

In [38]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = _Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [49]:
await ragas_faithfulness(result, "")

0.7857142857142857

In [41]:
async def ragas_response_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = _ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [50]:
await ragas_response_relevancy(result, "")

np.float64(0.8118286387276522)

In [52]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = _IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [53]:
await ragas_context_precision_id_based(result, reference_output)

0.2

In [54]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = _IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [64]:
await ragas_context_recall_id_based(result, reference_output)

0.5